# ProloGame - Pipeline Benchmark
Compares Prolog generation quality across model configurations and games.

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

import time
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import config
import src.pipeline.rule_generator as rule_gen
import src.pipeline.prolog_generator as prolog_gen

print("Setup complete.")

Setup complete.


---

## 1. Setup & Configs

In [ ]:
GAMES = [
    "Tic-Tac-Toe",
    "Connect Four",
    "Nim",
    "Gomoku",
    "Othello",
]

RUNS_PER_GAME = 3

CONFIGURATIONS = [
    {
        "name": "qwen3-coder:480b + design plan",
        "BACKEND_PROLOG_GENERATOR": "ollama",
        "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
        "PROLOG_USE_DESIGN_PLAN":   True,
    },
    {
        "name": "qwen3-coder:480b / no design plan",
        "BACKEND_PROLOG_GENERATOR": "ollama",
        "MODEL_PROLOG_GENERATOR":   "qwen3-coder:480b-cloud",
        "PROLOG_USE_DESIGN_PLAN":   False,
    },
    {
        "name": "gemma4:31b + design plan",
        "BACKEND_PROLOG_GENERATOR": "ollama",
        "MODEL_PROLOG_GENERATOR":   "gemma4:31b-cloud",
        "PROLOG_USE_DESIGN_PLAN":   True,
    }
]

RESULTS_PATH = "benchmark_results.csv"

print(f"{len(GAMES)} games x {len(CONFIGURATIONS)} configs x {RUNS_PER_GAME} runs = {len(GAMES) * len(CONFIGURATIONS) * RUNS_PER_GAME} total runs")

5 games x 3 configs x 3 runs = 45 total runs


---

## 1. Benchmark runner

In [ ]:
def apply_config(cfg: dict):
    """Apply a configuration dict to the config module."""
    for key, value in cfg.items():
        if key != "name" and hasattr(config, key):
            setattr(config, key, value)


def run_single(game: str, cfg_name: str) -> dict:
    """Run the full pipeline for one game and return a result dict."""
    result = {
        "game":          game,
        "config":        cfg_name,
        "success":       False,
        "retries":       0,
        "failed_checks": [],
        "error_stage":   None,
        "duration":      0.0,
    }

    t0 = time.time()

    try:
        # Stage 1: Rulebook
        ok, rulebook = rule_gen.generate_rulebook(game)
        if not ok:
            result["error_stage"] = "rulebook"
            return result

        # Stage 2: JSON
        ok, structured = rule_gen.rulebook_to_json(rulebook)
        if not ok:
            result["error_stage"] = "json"
            return result

        # Stage 3: Prolog - patch generate_prolog to track retries
        # We monkey-patch _validate_prolog to count attempts
        original_validate = prolog_gen._validate_prolog
        attempt_count = [0]
        last_errors = [[]]

        def tracking_validate(code):
            attempt_count[0] += 1
            valid, errors = original_validate(code)
            last_errors[0] = errors
            return valid, errors

        prolog_gen._validate_prolog = tracking_validate
        code, design_plan = prolog_gen.generate_prolog(structured)
        prolog_gen._validate_prolog = original_validate

        result["retries"] = attempt_count[0] - 1  # first attempt is not a retry

        if code is None:
            result["error_stage"]   = "prolog"
            result["failed_checks"] = last_errors[0]
            return result

        result["success"] = True

    except Exception as e:
        result["error_stage"] = f"exception: {type(e).__name__}: {e}"

    finally:
        result["duration"] = round(time.time() - t0, 1)

    return result


print("Runner defined.")

Runner defined.


In [ ]:
all_results = []
total = len(GAMES) * len(CONFIGURATIONS) * RUNS_PER_GAME
done  = 0

for cfg in CONFIGURATIONS:
    apply_config(cfg)
    cfg_name = cfg["name"]

    for game in GAMES:
        for run_idx in range(RUNS_PER_GAME):
            done += 1
            print(f"[{done}/{total}] {cfg_name} | {game} | run {run_idx + 1}", end=" ... ")

            result = run_single(game, cfg_name)
            result["run"] = run_idx + 1
            all_results.append(result)

            status = "OK" if result["success"] else f"FAIL ({result["error_stage"]})"
            print(f"{status} | {result["duration"]}s | retries: {result["retries"]}")

df = pd.DataFrame(all_results)
df["failed_checks"] = df["failed_checks"].apply(lambda x: " | ".join(x) if x else "")
df.to_csv(RESULTS_PATH, index=False)

print(f"\nDone. Results saved to {RESULTS_PATH}")
df.head()

[1/45] qwen3-coder:480b + design plan | Tic-Tac-Toe | run 1 ... OK | 57.7s | retries: 0
[2/45] qwen3-coder:480b + design plan | Tic-Tac-Toe | run 2 ... OK | 49.3s | retries: 0
[3/45] qwen3-coder:480b + design plan | Tic-Tac-Toe | run 3 ... OK | 60.7s | retries: 0
[4/45] qwen3-coder:480b + design plan | Connect Four | run 1 ... OK | 100.4s | retries: 0
[5/45] qwen3-coder:480b + design plan | Connect Four | run 2 ... OK | 38.7s | retries: 0
[6/45] qwen3-coder:480b + design plan | Connect Four | run 3 ... OK | 40.8s | retries: 0
[7/45] qwen3-coder:480b + design plan | Nim | run 1 ... FAIL (prolog) | 92.1s | retries: 4
[8/45] qwen3-coder:480b + design plan | Nim | run 2 ... FAIL (prolog) | 100.7s | retries: 4
[9/45] qwen3-coder:480b + design plan | Nim | run 3 ... OK | 129.0s | retries: 3
[10/45] qwen3-coder:480b + design plan | Gomoku | run 1 ... OK | 93.8s | retries: 0
[11/45] qwen3-coder:480b + design plan | Gomoku | run 2 ... OK | 68.6s | retries: 0
[12/45] qwen3-coder:480b + design pl

,game,config,success,retries,failed_checks,error_stage,duration,run
0,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,,None,57.7,1
1,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,,None,49.3,2
2,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,,None,60.7,3
3,Connect Four,qwen3-coder:480b + design plan,True,0,,None,100.4,1
4,Connect Four,qwen3-coder:480b + design plan,True,0,,None,38.7,2


In [ ]:
df = pd.read_csv(RESULTS_PATH)
print(f"Loaded {len(df)} results.")
df.head()

Loaded 45 results.


,game,config,success,retries,failed_checks,error_stage,duration,run
0,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,NaN,NaN,57.7,1
1,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,NaN,NaN,49.3,2
2,Tic-Tac-Toe,qwen3-coder:480b + design plan,True,0,NaN,NaN,60.7,3
3,Connect Four,qwen3-coder:480b + design plan,True,0,NaN,NaN,100.4,1
4,Connect Four,qwen3-coder:480b + design plan,True,0,NaN,NaN,38.7,2


---

## 2. Success rate per config

In [ ]:
success_rate = (
    df.groupby("config")["success"]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .rename(columns={"success": "success_rate"})
    .sort_values("success_rate", ascending=False)
)

fig = px.bar(
    success_rate,
    x="config",
    y="success_rate",
    text="success_rate",
    title="Success rate per configuration (%)",
    labels={"config": "Configuration", "success_rate": "Success rate (%)"},
    color="success_rate",
    color_continuous_scale="teal",
    range_y=[0, 100],
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(coloraxis_showscale=False, showlegend=False)
fig.show()

---

## 3. Success rate per game per config

In [ ]:
heatmap_data = (
    df.groupby(["game", "config"])["success"]
    .mean()
    .mul(100)
    .round(0)
    .unstack("config")
)

fig = px.imshow(
    heatmap_data,
    text_auto=True,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=100,
    title="Success rate (%) per game x configuration",
    labels={"color": "Success rate (%)"},
    aspect="auto",
)
fig.show()

---

## 4. Retry distribution per config

In [ ]:
fig = px.histogram(
    df[df["success"] == True],
    x="retries",
    color="config",
    barmode="group",
    title="Retry distribution (successful runs only)",
    labels={"retries": "Number of retries", "count": "Runs"},
    nbins=config.PROLOG_MAX_RETRIES + 1,
)
fig.show()

---

## 5. Validator chokepoints

In [ ]:
failed_df = df[(df["failed_checks"] != "") & (df["failed_checks"].notna())]

check_counts = {}
for _, row in failed_df.iterrows():
    for check in row["failed_checks"].split(" | "):
        check_label = check.split("]")[0].replace("[", "").strip() if "]" in check else check[:40]
        check_counts[check_label] = check_counts.get(check_label, 0) + 1

check_df = (
    pd.DataFrame(list(check_counts.items()), columns=["check", "count"])
    .sort_values("count", ascending=True)
)

fig = px.bar(
    check_df,
    x="count",
    y="check",
    orientation="h",
    title="Validator check failures (all failed runs)",
    labels={"count": "Number of failures", "check": "Check"},
    color="count",
    color_continuous_scale="reds",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

---

## 6. Average duration per config

In [ ]:
duration_df = (
    df.groupby("config")["duration"]
    .mean()
    .round(1)
    .reset_index()
    .sort_values("duration")
)

fig = px.bar(
    duration_df,
    x="config",
    y="duration",
    text="duration",
    title="Average pipeline duration per configuration (seconds)",
    labels={"config": "Configuration", "duration": "Avg. duration (s)"},
    color="duration",
    color_continuous_scale="blues",
)
fig.update_traces(texttemplate="%{text}s", textposition="outside")
fig.update_layout(coloraxis_showscale=False)
fig.show()

---

## 7. Summary

In [ ]:
summary = df.groupby("config").agg(
    success_rate=("success", lambda x: f"{x.mean()*100:.0f}%"),
    avg_retries=("retries", lambda x: f"{x.mean():.1f}"),
    avg_duration=("duration", lambda x: f"{x.mean():.0f}s"),
    total_runs=("success", "count"),
).reset_index()

summary.columns = ["Configuration", "Success rate", "Avg retries", "Avg duration", "Total runs"]
summary

,Configuration,Success rate,Avg retries,Avg duration,Total runs
0,gemma4:31b + design plan,80%,1.1,74s,15
1,qwen3-coder:480b + design plan,87%,0.8,79s,15
2,qwen3-coder:480b / no design plan,93%,0.8,103s,15
